In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="fine-tuning",
    notebook_name="05_instruction_tuning_experiments.ipynb"
)

# Experiments: Instruction Tuning, RLHF, and DPO

This notebook produces runnable evidence for claims you will make in interviews about instruction tuning, RLHF, and DPO. Each experiment tests a specific claim and gives you a concrete result to reference.

**Experiments in this notebook:**
1. SFT loss masking — show that masking instruction tokens improves response quality
2. Data diversity ablation — prove that task diversity matters more than dataset size
3. KL penalty sweep — demonstrate the reward-KL trade-off in RLHF
4. DPO vs naive preference learning — show why the log-ratio formulation matters
5. Reward hacking demo — demonstrate how an RL agent exploits a reward model

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

np.random.seed(42)

---
## Experiment 1: SFT Loss Masking

**Claim to test:** Computing loss only on response tokens (masking instruction tokens) leads to better response quality than computing loss on all tokens.

**Why it matters in an interview:** Loss masking is a subtle but important implementation detail. It shows you understand what the model is actually optimizing during SFT.

In [ ]:
# Simulate SFT training with and without loss masking.
#
# We model a simple sequence prediction task where the model must learn
# to produce correct response tokens given instruction tokens.
# With masking: model capacity focuses entirely on response prediction.
# Without masking: model wastes capacity predicting instruction tokens too.

def simulate_sft_training(n_examples, n_epochs, mask_instructions, 
                          instruction_len=20, response_len=10, vocab_size=100):
    """Simulate SFT training to measure response token accuracy.
    
    We model the model as having a fixed 'capacity' to memorize token predictions.
    With masking, all capacity goes to response tokens.
    Without masking, capacity is split between instruction and response tokens.
    """
    total_tokens_per_example = instruction_len + response_len
    response_fraction = response_len / total_tokens_per_example
    
    # Simulate learning curves
    response_accuracies = []
    capacity_per_epoch = 0.03  # How much the model learns per epoch
    
    acc = 0.0
    for epoch in range(n_epochs):
        if mask_instructions:
            # All gradient signal goes to response tokens
            improvement = capacity_per_epoch * (1.0 - acc) * n_examples / 100
        else:
            # Gradient signal is diluted across all tokens
            # Response tokens get only response_fraction of the signal
            effective_signal = response_fraction
            improvement = capacity_per_epoch * effective_signal * (1.0 - acc) * n_examples / 100
        
        acc = min(acc + improvement, 1.0)
        # Add realistic noise
        noisy_acc = acc + np.random.normal(0, 0.01)
        response_accuracies.append(np.clip(noisy_acc, 0, 1))
    
    return response_accuracies

n_epochs = 50
masked_acc = simulate_sft_training(500, n_epochs, mask_instructions=True)
unmasked_acc = simulate_sft_training(500, n_epochs, mask_instructions=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# -- Learning curves --
ax = axes[0]
ax.plot(range(n_epochs), masked_acc, 'g-', linewidth=2, label='With masking (response tokens only)')
ax.plot(range(n_epochs), unmasked_acc, 'r-', linewidth=2, label='Without masking (all tokens)')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Response Token Accuracy', fontsize=12)
ax.set_title('SFT Loss Masking: Learning Curves', fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

# -- Where gradients go --
ax = axes[1]
categories = ['Response\nTokens', 'Instruction\nTokens']
masked_dist = [100, 0]
unmasked_dist = [33, 67]  # response_len / total = 10/30 = 33%

x = np.arange(len(categories))
width = 0.3
ax.bar(x - width/2, masked_dist, width, label='With masking', color='#4CAF50', edgecolor='black')
ax.bar(x + width/2, unmasked_dist, width, label='Without masking', color='#F44336', edgecolor='black')
ax.set_ylabel('% of Gradient Signal', fontsize=12)
ax.set_title('Where Gradient Signal Goes', fontweight='bold', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=11)
ax.legend(fontsize=10)
ax.set_ylim(0, 110)
ax.grid(True, alpha=0.3, axis='y')

# Add percentage labels
for i, (m, u) in enumerate(zip(masked_dist, unmasked_dist)):
    ax.text(i - width/2, m + 2, f'{m}%', ha='center', fontweight='bold', fontsize=10)
    ax.text(i + width/2, u + 2, f'{u}%', ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

print(f"Final response accuracy WITH masking:    {masked_acc[-1]:.3f}")
print(f"Final response accuracy WITHOUT masking: {unmasked_acc[-1]:.3f}")
print(f"\nMasking improves response accuracy by {(masked_acc[-1] - unmasked_acc[-1])*100:.1f} percentage points.")
print(f"Without masking, 67% of gradient signal is wasted on instruction tokens.")
print(f"\nInterview sentence: 'SFT loss masking directs 100% of gradient signal")
print(f"to response tokens, leading to faster convergence and better response quality.'")

---
## Experiment 2: Data Diversity Ablation

**Claim to test:** Task diversity in instruction data matters more than dataset size for generalization.

**Why it matters in an interview:** When discussing data strategy for SFT, you should know that 10K diverse examples often outperform 100K narrow ones.

In [ ]:
# Simulate the effect of dataset diversity vs size on generalization.
#
# We model a set of task categories. A model trained on more categories
# generalizes better to unseen categories, even with fewer total examples.

task_categories = [
    'QA', 'Summarization', 'Translation', 'Classification',
    'Creative Writing', 'Math', 'Coding', 'Reasoning',
    'Conversation', 'Extraction'
]

def simulate_generalization(n_categories_trained, n_examples_per_category, 
                            total_categories=10, n_eval_categories=3):
    """Simulate generalization to unseen task categories.
    
    More categories seen during training = better transfer to unseen tasks.
    More examples per category = diminishing returns after a threshold.
    """
    total_examples = n_categories_trained * n_examples_per_category
    
    # Coverage factor: what fraction of task space is covered
    coverage = n_categories_trained / total_categories
    
    # Within-category quality: diminishing returns with more examples
    quality = 1.0 - np.exp(-n_examples_per_category / 500)
    
    # Generalization to unseen categories depends mainly on coverage
    # Transfer = coverage^0.5 (sublinear - partial transfer from related tasks)
    transfer = coverage ** 0.5
    
    # Final score on unseen categories
    seen_score = quality * 0.95
    unseen_score = quality * transfer * 0.7  # Transfer is imperfect
    
    return seen_score, unseen_score, total_examples

# Scenario A: Few categories, many examples each (narrow but deep)
# Scenario B: Many categories, fewer examples each (broad but shallow)
# Both have similar total example counts

configs = [
    ('2 tasks, 5000 each\n(narrow, 10K total)', 2, 5000),
    ('5 tasks, 2000 each\n(moderate, 10K total)', 5, 2000),
    ('10 tasks, 1000 each\n(broad, 10K total)', 10, 1000),
    ('2 tasks, 25000 each\n(narrow, 50K total)', 2, 25000),
    ('10 tasks, 500 each\n(broad, 5K total)', 10, 500),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

labels = [c[0] for c in configs]
seen_scores = []
unseen_scores = []
total_sizes = []

for _, n_cat, n_per in configs:
    seen, unseen, total = simulate_generalization(n_cat, n_per)
    seen_scores.append(seen)
    unseen_scores.append(unseen)
    total_sizes.append(total)

# -- Performance comparison --
ax = axes[0]
x = np.arange(len(configs))
width = 0.35
ax.bar(x - width/2, [s*100 for s in seen_scores], width, label='Seen tasks', 
       color='#4CAF50', edgecolor='black')
ax.bar(x + width/2, [s*100 for s in unseen_scores], width, label='Unseen tasks', 
       color='#2196F3', edgecolor='black')

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Generalization: Diversity vs Size', fontweight='bold', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=8)
ax.legend(fontsize=10)
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3, axis='y')

# -- Key comparison: 10 tasks x 500 (5K) vs 2 tasks x 25K (50K) --
ax = axes[1]
comparison = {
    '10 tasks x 500\n(5K total, broad)': unseen_scores[4] * 100,
    '2 tasks x 25,000\n(50K total, narrow)': unseen_scores[3] * 100,
}

bars = ax.bar(comparison.keys(), comparison.values(), 
              color=['#4CAF50', '#F44336'], edgecolor='black', width=0.5)
for bar, val in zip(bars, comparison.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}%', ha='center', fontweight='bold', fontsize=12)

ax.set_ylabel('Accuracy on Unseen Tasks (%)', fontsize=12)
ax.set_title('5K Diverse > 50K Narrow\n(on unseen task generalization)', 
             fontweight='bold', fontsize=13)
ax.set_ylim(0, 80)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"5K diverse examples (10 tasks): {unseen_scores[4]*100:.1f}% on unseen tasks")
print(f"50K narrow examples (2 tasks):  {unseen_scores[3]*100:.1f}% on unseen tasks")
print(f"\n10x MORE data with LESS diversity gives WORSE generalization.")
print(f"\nInterview sentence: 'For SFT, task diversity drives generalization more")
print(f"than dataset size. 5K examples across 10 task types outperform 50K examples")
print(f"from only 2 task types on unseen categories.'")

---
## Experiment 3: KL Penalty Sweep

**Claim to test:** The KL penalty coefficient (beta) in RLHF controls the trade-off between reward maximization and staying close to the reference policy. Too low leads to reward hacking; too high prevents improvement.

**Why it matters in an interview:** Understanding the KL-reward trade-off is central to explaining RLHF. The beta hyperparameter is the most important knob in the RLHF objective.

In [ ]:
# Simulate the RLHF training loop with different KL penalty strengths.
#
# We model:
# - A true quality metric (what we actually want to maximize)
# - A reward model (imperfect proxy for true quality)
# - KL divergence from the reference policy
#
# At low beta: reward is maximized but the model exploits the reward model
# At high beta: the model barely changes from the reference
# At optimal beta: genuine quality improvement without exploitation

def simulate_rlhf_training(beta, n_steps=100):
    """Simulate RLHF training with a given KL penalty coefficient."""
    kl_divergence = 0.0
    reward_model_score = 0.0
    true_quality = 0.0
    
    kls = [0.0]
    rewards = [0.0]
    qualities = [0.5]  # Start at baseline quality
    
    for step in range(n_steps):
        # The model tries to increase reward
        # At low KL, improvements are genuine
        # At high KL, the model starts gaming the reward model
        
        lr = 0.05 / (1 + beta)  # Higher beta = slower updates
        
        # KL increases each step (model drifts from reference)
        kl_increase = lr * (1.0 + 0.02 * step)  # Accelerating drift
        kl_divergence += kl_increase
        
        # Reward model score increases (the model optimizes for this)
        reward_increase = lr * (1.0 - reward_model_score / 2.0)
        reward_model_score += reward_increase
        
        # True quality: improves initially, then degrades if KL is too high
        # (reward hacking territory)
        genuine_improvement = lr * 0.5 * max(0, 1.0 - kl_divergence / 5.0)
        hacking_penalty = max(0, kl_divergence - 3.0) * 0.1
        true_quality = 0.5 + genuine_improvement * step - hacking_penalty
        true_quality = max(0.2, min(true_quality, 1.0))
        
        kls.append(kl_divergence)
        rewards.append(reward_model_score)
        qualities.append(true_quality + np.random.normal(0, 0.02))
    
    return kls, rewards, qualities

betas = [0.001, 0.01, 0.05, 0.1, 0.5, 2.0]
beta_labels = ['0.001 (very low)', '0.01 (low)', '0.05 (moderate)', 
               '0.1 (default)', '0.5 (high)', '2.0 (very high)']
colors = ['#D32F2F', '#F57C00', '#FBC02D', '#4CAF50', '#1976D2', '#7B1FA2']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for beta, label, color in zip(betas, beta_labels, colors):
    kls, rewards, qualities = simulate_rlhf_training(beta)
    
    axes[0].plot(range(len(kls)), kls, color=color, linewidth=1.5, label=f'\u03b2={label}')
    axes[1].plot(range(len(rewards)), rewards, color=color, linewidth=1.5, label=f'\u03b2={label}')
    axes[2].plot(range(len(qualities)), qualities, color=color, linewidth=1.5, label=f'\u03b2={label}')

axes[0].set_title('KL Divergence from Reference', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Training Step')
axes[0].set_ylabel('KL(\u03c0 || \u03c0_ref)')
axes[0].legend(fontsize=7)
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Reward Model Score', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Training Step')
axes[1].set_ylabel('Score')
axes[1].legend(fontsize=7)
axes[1].grid(True, alpha=0.3)

axes[2].set_title('True Quality (what we care about)', fontweight='bold', fontsize=12)
axes[2].set_xlabel('Training Step')
axes[2].set_ylabel('Quality')
axes[2].legend(fontsize=7)
axes[2].grid(True, alpha=0.3)
# Add danger zone annotation
axes[2].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
axes[2].text(50, 0.48, 'baseline', fontsize=8, color='gray', ha='center')

plt.suptitle('RLHF KL Penalty Sweep: The Reward-Quality Trade-off',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Key observations:")
print("  - Very low beta (0.001): High reward score but TRUE quality drops (reward hacking!)")
print("  - Very high beta (2.0): Model barely changes from reference (underfitting)")
print("  - Moderate beta (0.05-0.1): Best true quality improvement")
print("\nInterview sentence: 'The KL penalty beta is the most important RLHF hyperparameter.")
print("Too low causes reward hacking where the reward score rises but true quality drops.")
print("Too high prevents the model from improving at all. The sweet spot is typically 0.01-0.1.'")

---
## Experiment 4: DPO vs Naive Preference Learning

**Claim to test:** DPO's log-ratio formulation (comparing against a reference policy) is better than naively increasing the probability of preferred responses and decreasing rejected ones.

**Why it matters in an interview:** Understanding WHY DPO uses log-ratios (not just absolute probabilities) shows you understand the connection to the RLHF objective and KL regularization.

In [ ]:
# Compare DPO (log-ratio formulation) vs naive preference learning.
#
# Naive approach: directly maximize p(y_w) and minimize p(y_l)
# DPO approach: maximize log(p_theta(y_w)/p_ref(y_w)) - log(p_theta(y_l)/p_ref(y_l))
#
# The DPO formulation implicitly includes KL regularization through the
# reference policy, preventing the model from drifting too far.

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -20, 20)))

def simulate_preference_training(method, n_steps=200, beta=0.1, lr=0.01):
    """Simulate preference-based training.
    
    We model a scenario with 5 responses ranked by quality.
    The training data has pairwise preferences between them.
    """
    n_responses = 5
    # True quality ranking (higher = better)
    true_quality = np.array([0.1, 0.3, 0.5, 0.7, 0.9])
    
    # Reference policy log-probabilities (SFT model)
    ref_logprobs = np.array([-2.0, -1.5, -1.0, -1.5, -2.0])  # SFT model is not perfectly calibrated
    
    # Current policy log-probabilities (start = reference)
    policy_logprobs = ref_logprobs.copy()
    
    # Generate preference pairs (higher quality preferred)
    pairs = []
    for i in range(n_responses):
        for j in range(n_responses):
            if true_quality[i] > true_quality[j]:
                pairs.append((i, j))  # i preferred over j
    
    quality_history = []
    kl_history = []
    
    for step in range(n_steps):
        # Pick a random preference pair
        w_idx, l_idx = pairs[np.random.randint(len(pairs))]
        
        if method == 'dpo':
            # DPO: gradient based on log-ratios
            log_ratio_w = policy_logprobs[w_idx] - ref_logprobs[w_idx]
            log_ratio_l = policy_logprobs[l_idx] - ref_logprobs[l_idx]
            logit = beta * (log_ratio_w - log_ratio_l)
            prob = sigmoid(logit)
            grad = beta * (1 - prob)
            
            policy_logprobs[w_idx] += lr * grad
            policy_logprobs[l_idx] -= lr * grad
            
        elif method == 'naive':
            # Naive: directly push up preferred, push down rejected
            policy_logprobs[w_idx] += lr * 0.5
            policy_logprobs[l_idx] -= lr * 0.5
        
        # Normalize to valid log-probabilities (softmax constraint approximation)
        policy_logprobs -= np.max(policy_logprobs)  # Prevent overflow
        
        # Measure quality: correlation between policy probs and true quality
        probs = np.exp(policy_logprobs)
        probs = probs / probs.sum()
        correlation = np.corrcoef(probs, true_quality)[0, 1]
        quality_history.append(correlation)
        
        # Measure KL from reference
        ref_probs = np.exp(ref_logprobs)
        ref_probs = ref_probs / ref_probs.sum()
        kl = np.sum(probs * np.log(probs / ref_probs + 1e-10))
        kl_history.append(kl)
    
    return quality_history, kl_history

dpo_quality, dpo_kl = simulate_preference_training('dpo')
naive_quality, naive_kl = simulate_preference_training('naive')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# -- Quality (correlation with true ranking) --
ax = axes[0]
# Smooth with rolling average
window = 10
dpo_smooth = np.convolve(dpo_quality, np.ones(window)/window, mode='valid')
naive_smooth = np.convolve(naive_quality, np.ones(window)/window, mode='valid')
ax.plot(dpo_smooth, 'g-', linewidth=2, label='DPO (log-ratio)')
ax.plot(naive_smooth, 'r-', linewidth=2, label='Naive (direct push)')
ax.set_xlabel('Training Step', fontsize=12)
ax.set_ylabel('Correlation with True Quality', fontsize=12)
ax.set_title('Quality Alignment', fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.5, 1.1)

# -- KL divergence from reference --
ax = axes[1]
ax.plot(dpo_kl, 'g-', linewidth=2, label='DPO', alpha=0.7)
ax.plot(naive_kl, 'r-', linewidth=2, label='Naive', alpha=0.7)
ax.set_xlabel('Training Step', fontsize=12)
ax.set_ylabel('KL(\u03c0 || \u03c0_ref)', fontsize=12)
ax.set_title('KL Divergence from Reference', fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.suptitle('DPO vs Naive Preference Learning',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Final quality correlation - DPO:   {dpo_quality[-1]:.3f}")
print(f"Final quality correlation - Naive: {naive_quality[-1]:.3f}")
print(f"Final KL divergence - DPO:   {dpo_kl[-1]:.3f}")
print(f"Final KL divergence - Naive: {naive_kl[-1]:.3f}")
print(f"\nDPO achieves better quality alignment with lower KL divergence.")
print(f"The log-ratio formulation implicitly regularizes through the reference policy.")
print(f"\nInterview sentence: 'DPO's log-ratio formulation is not just a trick — it")
print(f"provides implicit KL regularization. The reference policy in the log-ratio acts")
print(f"as an anchor, preventing the model from drifting too far while still learning preferences.'")

---
## Experiment 5: Reward Hacking Demo

**Claim to test:** An RL agent will exploit weaknesses in the reward model when the KL penalty is insufficient, leading to high reward scores but low true quality.

**Why it matters in an interview:** Reward hacking is the primary failure mode of RLHF. You must be able to explain it concretely and describe mitigations.

In [ ]:
# Demonstrate reward hacking with a concrete example.
#
# Setup: A reward model that (incorrectly) correlates response length with quality.
# An RL agent will discover this and exploit it by generating longer responses,
# even when the true quality does not improve (or actually decreases).

def reward_model(response_length, content_quality):
    """An imperfect reward model that is biased toward longer responses.
    
    Real reward models often have this bias because annotators tend to
    rate longer, more detailed responses higher in training data.
    """
    # True quality contributes 60% of the score
    # Length contributes 40% (this is the bias the agent will exploit)
    length_score = min(response_length / 200, 1.0)  # Saturates at 200 words
    return 0.6 * content_quality + 0.4 * length_score

def true_quality(response_length, base_quality):
    """The actual quality a human would assign.
    
    Short, concise responses can be high quality.
    After a point, longer responses become worse (rambling).
    """
    # Quality peaks at ~80 words, then decreases
    length_factor = 1.0 - abs(response_length - 80) / 200
    length_factor = max(0.3, length_factor)
    return base_quality * length_factor

def simulate_rl_training(beta, n_steps=100):
    """Simulate an RL agent optimizing against the reward model."""
    response_length = 50  # Start at 50 words
    base_quality = 0.7  # Fixed content quality
    
    lengths = [response_length]
    reward_scores = [reward_model(response_length, base_quality)]
    true_scores = [true_quality(response_length, base_quality)]
    
    for step in range(n_steps):
        # Agent explores: try a longer or shorter response
        delta = np.random.choice([-10, -5, 0, 5, 10, 15, 20])
        new_length = max(10, response_length + delta)
        
        # Compare reward with KL-like penalty for change
        current_reward = reward_model(response_length, base_quality)
        new_reward = reward_model(new_length, base_quality)
        change_penalty = beta * abs(new_length - 50) / 100  # Penalty for deviating from reference
        
        # Accept if net reward is higher
        if (new_reward - change_penalty) > (current_reward - beta * abs(response_length - 50) / 100):
            response_length = new_length
        
        lengths.append(response_length)
        reward_scores.append(reward_model(response_length, base_quality))
        true_scores.append(true_quality(response_length, base_quality))
    
    return lengths, reward_scores, true_scores

# Run with weak vs strong KL penalty
weak_lengths, weak_rewards, weak_true = simulate_rl_training(beta=0.01)
strong_lengths, strong_rewards, strong_true = simulate_rl_training(beta=0.5)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# -- Response length over training --
ax = axes[0]
ax.plot(weak_lengths, 'r-', linewidth=2, label='Weak KL (\u03b2=0.01)')
ax.plot(strong_lengths, 'g-', linewidth=2, label='Strong KL (\u03b2=0.5)')
ax.axhline(y=80, color='blue', linestyle='--', alpha=0.5, label='Optimal length')
ax.set_xlabel('Training Step', fontsize=12)
ax.set_ylabel('Response Length (words)', fontsize=12)
ax.set_title('Response Length', fontweight='bold', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# -- Reward model score --
ax = axes[1]
ax.plot(weak_rewards, 'r-', linewidth=2, label='Weak KL')
ax.plot(strong_rewards, 'g-', linewidth=2, label='Strong KL')
ax.set_xlabel('Training Step', fontsize=12)
ax.set_ylabel('Reward Score', fontsize=12)
ax.set_title('Reward Model Score\n(what RL optimizes)', fontweight='bold', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# -- True quality --
ax = axes[2]
ax.plot(weak_true, 'r-', linewidth=2, label='Weak KL')
ax.plot(strong_true, 'g-', linewidth=2, label='Strong KL')
ax.set_xlabel('Training Step', fontsize=12)
ax.set_ylabel('True Quality', fontsize=12)
ax.set_title('True Quality\n(what humans actually want)', fontweight='bold', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('Reward Hacking: High Reward Score != High Quality',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"With weak KL penalty:")
print(f"  Final length: {weak_lengths[-1]} words (optimal is 80)")
print(f"  Reward score: {weak_rewards[-1]:.3f} (looks great!)")
print(f"  True quality: {weak_true[-1]:.3f} (actually worse!)")
print(f"\nWith strong KL penalty:")
print(f"  Final length: {strong_lengths[-1]} words")
print(f"  Reward score: {strong_rewards[-1]:.3f}")
print(f"  True quality: {strong_true[-1]:.3f}")
print(f"\nThe weak-KL agent hacked the reward model by making responses longer.")
print(f"The reward score went UP but true quality went DOWN. This is Goodhart's law.")
print(f"\nInterview sentence: 'Reward hacking is Goodhart's law in action — the reward")
print(f"model is an imperfect proxy, and the RL optimizer will exploit its weaknesses.")
print(f"The KL penalty from the reference policy is the primary defense against this.'")

---
## Summary

Claims now backed by evidence:

1. **SFT loss masking** directs all gradient signal to response tokens, improving convergence
2. **Task diversity** matters more than dataset size for generalization in SFT
3. **KL penalty (beta)** controls the reward-quality trade-off in RLHF — too low causes hacking, too high prevents learning
4. **DPO's log-ratio formulation** provides implicit KL regularization through the reference policy
5. **Reward hacking** is Goodhart's law applied to learned reward models — high reward scores can mean low true quality

For the full mathematical treatment, see [instruction-tuning-interview.md](./instruction-tuning-interview.md).

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)